# Compute walking speed from heel markers
Look at the coordinates of the heel markers from c3d file and compute walking speed from them.
Assume that c3d file only contains heel markers.

In [ ]:
import numpy as np
import ezc3d
import plotly.graph_objects as go

In [ ]:
DATA_PEFIX = '/home/andrey/scratch/walking_speed'
DATA_FILES = {'paljain jaloin': ['2026_02_12_vidvek_seur_EV/2026_02_12_vidvek_seur_EV11.c3d',
                                 '2026_02_12_vidvek_seur_EV/2026_02_12_vidvek_seur_EV12.c3d',
                                 '2026_02_12_vidvek_seur_EV/2026_02_12_vidvek_seur_EV13.c3d',
                                 '2026_02_12_vidvek_seur_EV/2026_02_12_vidvek_seur_EV14.c3d',
                                 '2026_02_12_vidvek_seur_EV/2026_02_12_vidvek_seur_EV15.c3d'],
              'tuet':           ['2026_02_12_vidvek_seur_tuet_EV/2026_02_12_vidvek_seur_tuet_EV05.c3d',
                                 '2026_02_12_vidvek_seur_tuet_EV/2026_02_12_vidvek_seur_tuet_EV06.c3d',
                                 '2026_02_12_vidvek_seur_tuet_EV/2026_02_12_vidvek_seur_tuet_EV15.c3d',
                                 '2026_02_12_vidvek_seur_tuet_EV/2026_02_12_vidvek_seur_tuet_EV16.c3d']}


In [ ]:
c3d_file = ezc3d.c3d(DATA_PEFIX + '/' + DATA_FILES['tuet'][0])

In [ ]:
sfreq = c3d_file['header']['points']['frame_rate']
n_samps = c3d_file['data']['points'].shape[2]

t = np.arange(n_samps)/sfreq
# Assume the markers we are interested in are the first two points (0:2)
y = c3d_file['data']['points'][:,0:2,:] / 1000

## Restrict ourselves by 25-th and 75-th percentiles

In [ ]:
mean_dist = y[1,:,:].mean(axis=0)
i25 = int(np.percentile(np.arange(n_samps), 25))
i75 = int(np.percentile(np.arange(n_samps), 75))

v = abs((mean_dist[i75] - mean_dist[i25]) / (t[i75] - t[i25]))

## Plot progress in walking direction

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=t[i25:i75], y=y[1,0,i25:i75], name='m1'))
fig.add_trace(go.Scatter(x=t[i25:i75], y=y[1,1,i25:i75], name='m2'))
fig.add_trace(go.Scatter(x=[t[i25], t[i75]], y=[mean_dist[i25], mean_dist[i75]], name=f'fit ({v:.2f} m/s)'))


## See how much we are movin sideways

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=t[i25:i75], y=y[0,0,i25:i75], name='m1'))
fig.add_trace(go.Scatter(x=t[i25:i75], y=y[0,1,i25:i75], name='m2'))

## Compute the speeds over all the trials

In [ ]:
res = {}

for cond in ['paljain jaloin', 'tuet']:
    res[cond] = []
    for fname in DATA_FILES[cond]:
        c3d_file = ezc3d.c3d(DATA_PEFIX + '/' + fname)
        sfreq = c3d_file['header']['points']['frame_rate']
        n_samps = c3d_file['data']['points'].shape[2]

        t = np.arange(n_samps)/sfreq
        y = c3d_file['data']['points'][:,0:2,:] / 1000

        mean_dist = y[1,:,:].mean(axis=0)
        i25 = int(np.percentile(np.arange(n_samps), 25))
        i75 = int(np.percentile(np.arange(n_samps), 75))

        v = abs((mean_dist[i75] - mean_dist[i25]) / (t[i75] - t[i25]))
        res[cond].append(v)

In [ ]:
res['paljain jaloin']

In [ ]:
res['tuet']

In [ ]:
fig = go.Figure()
fig.add_trace(go.Box(y=res['paljain jaloin'], name='paljain jaloin',
                marker_color = 'indianred'))
fig.add_trace(go.Box(y=res['tuet'], name = 'tuet',
                marker_color = 'lightseagreen'))

fig.show()
